In [ ]:
# directories
pdfs_dir = 'C:\\Users\\samwt\\Documents\\ExtendedEssay\\data\\raw\\rowells_directories\\pdfs'
ocr_text_dir = 'C:\\Users\\samwt\\Documents\\ExtendedEssay\\data\\raw\\rowells_directories\\ocr_text'
extracted_text_dir = 'C:\\Users\\samwt\\Documents\\ExtendedEssay\\data\\raw\\rowells_directories\\extracted_csv'

In [ ]:
# new merger

import pandas as pd
import os
from pathlib import Path
from difflib import SequenceMatcher
import re
from collections import defaultdict

DAYS_OF_WEEK = ['sundays', 'mondays', 'tuesdays', 'wednesdays', 'thursdays', 'fridays', 'saturdays']

# Precompile regex patterns for day removal and normalization
_DAYS_PATTERN = re.compile('|'.join(re.escape(day) for day in DAYS_OF_WEEK), re.IGNORECASE)
_NORMALIZE_PATTERN = re.compile(r"[.,'\s]")

def normalize_text(s):
    """Normalize text for matching: lowercase, strip whitespace, remove punctuation."""
    if pd.isna(s):
        return ""
    return _NORMALIZE_PATTERN.sub("", str(s).lower().strip())

def normalize_text_no_days(s):
    """Normalize text and also remove days of the week."""
    if pd.isna(s):
        return ""
    text = _DAYS_PATTERN.sub("", str(s).lower().strip())
    return _NORMALIZE_PATTERN.sub("", text)

def similarity(a, b):
    """Calculate similarity ratio between two strings (0 to 1)."""
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()

def is_fuzzy_match(town1, name1, town2, name2, threshold=0.90):
    """
    Check if two newspaper records match using fuzzy matching.
    More strict: requires high similarity on both fields.
    """
    town_sim = similarity(town1, town2)
    name_sim = similarity(name1, name2)
    
    if town_sim >= threshold and name_sim >= threshold:
        return True
    if town_sim == 1.0 and name_sim >= 0.85:
        return True
    if name_sim == 1.0 and town_sim >= 0.85:
        return True
    return False

def remove_days_from_name(name):
    """Remove days of the week from a newspaper name, preserving original formatting."""
    if pd.isna(name):
        return ""
    result = _DAYS_PATTERN.sub('', str(name))
    # Clean up extra spaces
    return ' '.join(result.split()).strip()

def find_best_match(town, name, position_pct, existing_records_flat, existing_records_by_letter,
                    existing_records_no_days, current_established=None, established_lookup=None,
                    threshold=0.90):
    """
    Find the best matching key from existing records.
    First tries exact match, then fuzzy match for towns with same first letter,
    then tries again with days of week removed from BOTH current and existing names.
    Uses lower threshold (80%) if established dates match.
    
    Returns tuple: (matched_key or None, matched_via_days_removal: bool)
    """
    town_norm = normalize_text(town)
    name_norm = normalize_text(name)
    name_norm_no_days = normalize_text_no_days(name)
    
    # First: try exact match (using flat dict)
    exact_key = (town_norm, name_norm)
    if exact_key in existing_records_flat:
        return exact_key, False
    
    # Get first letter of town for filtering
    town_first_letter = town_norm[0] if town_norm else ""
    if not town_first_letter:
        return None, False
    
    # Get candidates with same first letter (pre-indexed)
    candidates = existing_records_by_letter.get(town_first_letter, {})
    if not candidates:
        return None, False
    
    # Second: try fuzzy match, only considering towns with same first letter
    # Now also compares with days removed from BOTH names
    best_match = None
    best_score = 0
    matched_via_days = False
    
    for (ex_town, ex_name), ex_position in candidates.items():
        # Check if established dates match for lower threshold
        effective_threshold = threshold
        if current_established and established_lookup:
            ex_established = established_lookup.get((ex_town, ex_name))
            if ex_established and str(current_established).strip() == str(ex_established).strip():
                effective_threshold = 0.80
        
        # Try standard fuzzy match first
        if is_fuzzy_match(town_norm, name_norm, ex_town, ex_name, effective_threshold):
            score = similarity(town_norm, ex_town) + similarity(name_norm, ex_name)
            if score > best_score:
                best_score = score
                best_match = (ex_town, ex_name)
                matched_via_days = False
            continue  # Found a match, no need to try days-removed for this record
        
        # Try with days removed from BOTH current and existing names
        # Use pre-computed no-days version
        ex_name_no_days = existing_records_no_days.get((ex_town, ex_name), ex_name)
        if is_fuzzy_match(town_norm, name_norm_no_days, ex_town, ex_name_no_days, effective_threshold):
            score = similarity(town_norm, ex_town) + similarity(name_norm_no_days, ex_name_no_days)
            if score > best_score:
                best_score = score
                best_match = (ex_town, ex_name)
                matched_via_days = True
    
    return best_match, matched_via_days

def load_and_tag_csvs(directory):
    """Load all CSVs from directory, tag with year, and split into pre/post 1877."""
    csv1_frames = []
    csv2_frames = []
    
    for file in Path(directory).glob("*.csv"):
        filename = file.stem
        year = None
        
        match = re.search(r'(1[89]\d{2})', filename)
        if match:
            year = int(match.group(1))
        
        if year is None:
            print(f"Warning: Could not extract year from {file.name}, skipping...")
            continue
        
        df = pd.read_csv(file, encoding='utf-8', on_bad_lines='skip')
        df['_year'] = year
        
        print(f"Loaded {file.name} (year {year}): {len(df)} records")
        
        if year <= 1876:
            csv1_frames.append(df)
        else:
            csv2_frames.append(df)
    
    return csv1_frames, csv2_frames

def process_dataframe(df, year, has_state=False):
    if 'raw_text' in df.columns:
        df = df.drop(columns=['raw_text'])
    
    df.columns = [c.lower().strip() for c in df.columns]
    
    # Normalize variant column names to standard names
    column_aliases = {
        'newspaper': 'newspaper_name',
        'political': 'political_affiliation',
    }
    df = df.rename(columns={k: v for k, v in column_aliases.items() if k in df.columns})
    
    # ... rest of function
    """Process a single dataframe: standardize and prepare for merging."""
    if 'raw_text' in df.columns:
        df = df.drop(columns=['raw_text'])
    
    df.columns = [c.lower().strip() for c in df.columns]
    
    data_cols = ['frequency', 'political_affiliation', 'subscription_price', 
                 'established', 'editor', 'publisher', 'circulation']
    
    rename_map = {}
    for col in data_cols:
        if col in df.columns:
            rename_map[col] = f"{year} {col}"
    
    df = df.rename(columns=rename_map)
    return df

def merge_newspapers_core(all_frames):
    """Core merge logic used by both full and test functions."""
    merged_records = {}
    # Flat dict for exact lookups
    record_positions_flat = {}
    # Index by first letter of town for faster fuzzy lookups
    record_positions_by_letter = defaultdict(dict)
    # Pre-computed normalized names with days removed
    existing_records_no_days = {}
    original_names = {}
    established_lookup = {}  # Track established dates for each record
    
    print(f"Processing {len(all_frames)} files...")
    print("Strategy: exact match first, then fuzzy match (same first letter, 90% similarity),")
    print("          with days of week removed from BOTH current and existing names,")
    print("          80% threshold if established dates match\n")
    
    for df, year, has_state in all_frames:
        total_rows = len(df)
        print(f"  Processing year {year} ({total_rows} records)...")
        matches_found = 0
        new_records = 0
        
        # Collect new records for this year, add to main dict after processing
        year_new_keys = []
        
        # Find the established column for this year
        established_col = f"{year} established"
        
        for idx, row in df.iterrows():
            row_num = df.index.get_loc(idx)
            position_pct = row_num / max(total_rows - 1, 1)
            
            town = row.get('town', '')
            name = row.get('newspaper_name', '')
            state = row.get('state', '') if has_state else ''
            current_established = row.get(established_col, None)
            
            town_norm = normalize_text(town)
            name_norm = normalize_text(name)
            
            if not town_norm or not name_norm:
                continue
            
            # Only match against records from previous years
            existing_key, matched_via_days = find_best_match(
                town, name, position_pct,
                record_positions_flat,
                record_positions_by_letter,
                existing_records_no_days,
                current_established=current_established,
                established_lookup=established_lookup,
                threshold=0.90
            )
            
            if existing_key:
                key = existing_key
                matches_found += 1
                old_pos = record_positions_flat[key]
                new_pos = (old_pos + position_pct) / 2
                record_positions_flat[key] = new_pos
                # Update indexed version too
                first_letter = key[0][0] if key[0] else ""
                if first_letter:
                    record_positions_by_letter[first_letter][key] = new_pos
                
                # If matched via days removal, update the stored name to remove days
                if matched_via_days:
                    old_town, old_name, old_state = original_names[key]
                    cleaned_name = remove_days_from_name(old_name)
                    original_names[key] = (old_town, cleaned_name, old_state)
            else:
                key = (town_norm, name_norm)
                merged_records[key] = {}
                original_names[key] = (town, name, state)
                # Queue this to be added after processing this year
                year_new_keys.append((key, position_pct, current_established))
                new_records += 1
            
            if state and not original_names[key][2]:
                original_names[key] = (original_names[key][0], original_names[key][1], state)
            
            year_cols = [c for c in row.index if c.startswith(f"{year} ")]
            for col in year_cols:
                merged_records[key][col] = row[col]
        
        # Now add this year's new records to positions for next year's matching
        for key, pos, estab in year_new_keys:
            record_positions_flat[key] = pos
            # Index by first letter
            first_letter = key[0][0] if key[0] else ""
            if first_letter:
                record_positions_by_letter[first_letter][key] = pos
            # Pre-compute no-days version
            existing_records_no_days[key] = normalize_text_no_days(key[1])
            if estab:
                established_lookup[key] = estab
        
        print(f"    -> {matches_found} matched to existing, {new_records} new records")
    
    print(f"\nTotal unique newspapers found: {len(merged_records)}")
    
    rows = []
    for key, data in merged_records.items():
        town, name, state = original_names[key]
        row = {'state': state, 'town': town, 'newspaper_name': name}
        row.update(data)
        rows.append(row)
    
    result = pd.DataFrame(rows)
    
    id_cols = ['state', 'town', 'newspaper_name']
    year_cols = [c for c in result.columns if c not in id_cols]
    
    def sort_key(col):
        parts = col.split(' ', 1)
        if len(parts) == 2 and parts[0].isdigit():
            return (int(parts[0]), parts[1])
        return (9999, col)
    
    year_cols = sorted(year_cols, key=sort_key)
    final_cols = id_cols + year_cols
    result = result[final_cols]
    result = result.sort_values(['state', 'town', 'newspaper_name'])
    
    return result

def prepare_frames(directory, max_years=None):
    """Load and prepare frames, optionally limiting to first N years."""
    csv1_frames, csv2_frames = load_and_tag_csvs(directory)
    
    if not csv1_frames and not csv2_frames:
        print("No CSV files found!")
        return None
    
    all_frames_raw = []
    
    for df in csv1_frames:
        year = df['_year'].iloc[0]
        all_frames_raw.append((df, year, False))
    
    for df in csv2_frames:
        year = df['_year'].iloc[0]
        all_frames_raw.append((df, year, True))
    
    all_frames_raw.sort(key=lambda x: x[1])
    
    if max_years is not None:
        all_frames_raw = all_frames_raw[:max_years]
        years_processing = [f[1] for f in all_frames_raw]
        print(f"\nTEST MODE: Processing only first {max_years} years: {years_processing}\n")
    
    all_frames = []
    for df, year, has_state in all_frames_raw:
        df = df.drop(columns=['_year'])
        df = process_dataframe(df, year, has_state=has_state)
        all_frames.append((df, year, has_state))
    
    return all_frames

def merge_newspapers_fuzzy(directory):
    """Main function to merge all newspaper CSVs with fuzzy matching."""
    all_frames = prepare_frames(directory)
    if all_frames is None:
        return None
    return merge_newspapers_core(all_frames)


if __name__ == "__main__":
    
    directory = extracted_text_dir  # Change this to your directory containing the CSVs
    
    print(f"Processing CSVs from: {directory}")
    print("=" * 60)
    
    result = merge_newspapers_fuzzy(directory)
    
    if result is not None:
        output_path = "master.csv"
        result.to_csv(output_path, index=False)
        print(f"\nSuccess! Output saved to: {output_path}")
        print(f"Total newspapers: {len(result)}")
        print(f"\nColumns in output:")
        for col in result.columns:
            print(f"  - {col}")
    else:
        print("Failed to create merged CSV.")

Processing CSVs from: data\Newspaper Directory Excel
Loaded Rowell 1869.csv (year 1869): 3072 records
Loaded Rowell 1871.csv (year 1871): 5878 records
Loaded Rowell 1872.csv (year 1872): 6241 records
Loaded Rowell 1873.csv (year 1873): 6550 records
Loaded Rowell 1876.csv (year 1876): 7825 records
Loaded Rowell 1877.csv (year 1877): 7274 records
Loaded Rowell 1878.csv (year 1878): 7548 records
Loaded Rowell 1879.csv (year 1879): 7785 records
Loaded Rowell 1880.csv (year 1880): 8561 records
Loaded Rowell 1882.csv (year 1882): 10435 records
Loaded Rowell 1883.csv (year 1883): 10171 records
Loaded Rowell 1884.csv (year 1884): 11484 records
Loaded Rowell 1885.csv (year 1885): 12282 records
Loaded Rowell 1890.csv (year 1890): 15629 records
Processing 14 files...
Strategy: exact match first, then fuzzy match (same first letter, 90% similarity),
          with days of week removed from BOTH current and existing names,
          80% threshold if established dates match

  Processing year 1869 (

In [ ]:
# descriptive stats

import pandas as pd

df = pd.read_csv("data/processed/master.csv")

# Get year columns (those starting with a 4-digit year)
year_prefixes = set()
for col in df.columns:
    parts = col.split(' ', 1)
    if len(parts) == 2 and parts[0].isdigit() and len(parts[0]) == 4:
        year_prefixes.add(parts[0])

years = sorted(year_prefixes)
print(f"Years in dataset: {years}\n")

# For each row, count how many years have any data
def count_years_with_data(row):
    count = 0
    for year in years:
        year_cols = [c for c in df.columns if c.startswith(f"{year} ")]
        if any(pd.notna(row[c]) for c in year_cols):
            count += 1
    return count

df['years_of_data'] = df.apply(count_years_with_data, axis=1)

# Summary
print("Newspapers by number of years with data:")
print("-" * 40)
counts = df['years_of_data'].value_counts().sort_index()
for num_years, count in counts.items():
    print(f"  {num_years} year(s): {count} newspapers")

print(f"\nTotal newspapers: {len(df)}")

C:\Users\samwt\AppData\Local\Temp\ipykernel_5792\3684759370.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/master.csv")


Years in dataset: ['1869', '1871', '1872', '1873', '1876', '1877', '1878', '1879', '1880', '1882', '1883', '1884', '1885', '1890']

Newspapers by number of years with data:
----------------------------------------
  0 year(s): 1277 newspapers
  1 year(s): 28675 newspapers
  2 year(s): 6484 newspapers
  3 year(s): 3584 newspapers
  4 year(s): 2496 newspapers
  5 year(s): 1467 newspapers
  6 year(s): 949 newspapers
  7 year(s): 756 newspapers
  8 year(s): 747 newspapers
  9 year(s): 608 newspapers
  10 year(s): 521 newspapers
  11 year(s): 525 newspapers
  12 year(s): 565 newspapers
  13 year(s): 325 newspapers
  14 year(s): 65 newspapers

Total newspapers: 49044


In [ ]:
# CLEANING UP master.csv 

import pandas as pd

# Load the data
df = pd.read_csv('data/processed/master.csv')

# Define the years we're tracking
years = [1869, 1871, 1872, 1873, 1876, 1877, 1878, 1879, 1880, 1882, 1883, 1884, 1885, 1890]

def levenshtein_distance(s1, s2):
    """Calculate the Levenshtein distance between two strings."""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        curr_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = prev_row[j + 1] + 1
            deletions = curr_row[j] + 1
            substitutions = prev_row[j] + (c1 != c2)
            curr_row.append(min(insertions, deletions, substitutions))
        prev_row = curr_row
    return prev_row[-1]

# Clean publisher columns - remove trailing ', editorand' or ', editor and' (with fuzzy matching)
def clean_publisher(val):
    if pd.isna(val):
        return val
    val = str(val).strip()
    lower_val = val.lower()
    
    # Check for ', editor and' first (exact match)
    if lower_val.endswith(', editor and'):
        return val[:-len(', editor and')]
    
    # Check for ', editorand' with fuzzy matching (1 char tolerance)
    # Look for ', ' followed by something close to 'editorand'
    if ', ' in lower_val:
        last_comma_idx = lower_val.rfind(', ')
        suffix = lower_val[last_comma_idx + 2:]  # text after ', '
        if levenshtein_distance(suffix, 'editorand') <= 1:
            return val[:last_comma_idx]
        if levenshtein_distance(suffix, 'editor and') <= 1:
            return val[:last_comma_idx]
    
    return val

# Apply cleaning to all publisher columns
publisher_cols = [f'{year} publisher' for year in years]
changes_made = 0

for col in publisher_cols:
    if col in df.columns:
        original = df[col].copy()
        df[col] = df[col].apply(clean_publisher)
        changes_made += (original != df[col]).sum()

print(f"Cleaned {changes_made} publisher entries")
print("Removed trailing ', editorand' and ', editor and'")

# Save back to master.csv
df.to_csv('data/processed/master.csv', index=False)
print("Saved to data/processed/master.csv")

C:\Users\samwt\AppData\Local\Temp\ipykernel_11944\1346512040.py:6: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/master.csv')


Cleaned 570145 publisher entries
Removed trailing ', editorand' and ', editor and'
Saved to data/master.csv
